# Parameteriser

Brenda API (requires user to create login)  
https://www.brenda-enzymes.org/soap.php

In [ ]:
# %pip install zeep
# %pip install ptitprince

In [ ]:
import hashlib
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy.stats import gaussian_kde
from sspipe import p
from zeep import Client

from typing import Type, TypeVar
import qtbutils as qu
from pathlib import Path
from typing import cast
from typing import Callable
from functools import partial

TexPath = qu.default_path("tex")


@dataclass
class BrendaType: ...


DataClass = TypeVar("DataClass", bound=BrendaType)


@dataclass
class Km(BrendaType):
    value: float
    substrate: str
    organism: str
    commentary: str | None
    literature: list[str]


@dataclass
class Sequence(BrendaType):
    first_accession_code: str
    naa: int
    sequence: str
    source: str
    organism: str
    id: str


def normalise(x: np.ndarray) -> np.ndarray:
    return x / np.sum(x)


def data_coordinates_of_width(ax, width):
    return ax.transData.inverted().transform(ax.transAxes.transform([width, 0]))[0]


def data_coordinates_of_height(ax, height):
    return ax.transData.inverted().transform(ax.transAxes.transform([0, height]))[1]


def data_to_display(ax, val):
    return ax.transData.transform(val)


def display_to_data(ax, val):
    return ax.transData.inverted().transform(val)


def axes_to_display(ax, val):
    return ax.transAxes.transform(val)


def display_to_axes(ax, val):
    return ax.transAxes.inverted().transform(val)


def x_to_data(ax, val):
    return ax.transLimits.inverted().transform((val, 0))[0]


def y_to_data(ax, val):
    return ax.transLimits.inverted().transform((0, val))[1]


def add_boxplot(ax, data, color="C0", offset: int = 0):
    _d = data.describe()
    iqr = _d["75%"] - _d["25%"]
    height = y_to_data(ax, 0.05)

    # Box
    ax.add_artist(
        Rectangle(
            (_d["25%"], offset * height),
            width=_d["75%"] - _d["25%"],
            height=height,
            facecolor=color,
            linewidth=1.5,
            alpha=0.7,
        )
    )

    # Bars
    ax.add_artist(
        Line2D(
            xdata=[data.median()],
            ydata=[offset * height, offset * height + height],
            color="white",
        )
    )

    # Whiskers
    ax.add_artist(
        Line2D(
            xdata=[max(_d["25%"] - 1.5 * iqr, ax.get_xlim()[0]), _d["25%"]],
            ydata=[offset * height + height / 2],
            color=color,
            alpha=0.7,
        )
    )
    ax.add_artist(
        Line2D(
            xdata=[_d["75%"], _d["75%"] + 1.5 * iqr],
            ydata=[offset * height + height / 2],
            color=color,
            alpha=0.7,
        )
    )


with open(".env") as fp:
    cred: dict[str, str] = dict(
        line.split("=", maxsplit=1) for line in fp.read().strip().split("\n")
    )

In [ ]:
@dataclass
class Brenda:
    email: str
    password: str
    tmp_dir: Path = Path(".") / "tmp"
    wsdl: str = "https://www.brenda-enzymes.org/soap/brenda_zeep.wsdl"

    def __post_init__(self) -> None:
        self.password = hashlib.sha256(self.password.encode("utf-8")).hexdigest()
        self.tmp_dir.mkdir(exist_ok=True, parents=True)

    def _cache_result(
        self,
        filename: Path,
        obj_type: Type[DataClass],
        download_fn: Callable[..., list[DataClass]],
        verbose: bool = False,
    ) -> list[DataClass]:
        if filename.exists():
            if verbose:
                print("Using cached data")
            with open(filename, "r", encoding="utf-8") as fp:
                data = [obj_type(**i) for i in json.load(fp)]
        else:
            if verbose:
                print("Downloading data")
            data = download_fn()
            with open(filename, "w", encoding="utf-8") as fp:
                json.dump([asdict(i) for i in data], fp)

        return data

    def get_km(self, ec_number: str, verbose: bool = False) -> pd.DataFrame:
        def download() -> list[Km]:
            return [
                Km(
                    value=float(res["kmValue"]),
                    substrate=res["substrate"],
                    organism=res["organism"],
                    commentary=res["commentary"],
                    literature=res["literature"],
                )
                for res in Client(self.wsdl).service.getKmValue(
                    self.email,
                    self.password,
                    f"ecNumber*{ec_number}",
                    "organism*",
                    "kmValue*",
                    "kmValueMaximum*",
                    "substrate*",
                    "commentary*",
                    "ligandStructureId*",
                    "literature*",
                )
            ]

        return pd.DataFrame(
            self._cache_result(
                filename=self.tmp_dir / f"km-{ec_number}.json",
                obj_type=Km,
                download_fn=download,
                verbose=verbose,
            )
        )

    def get_sequences(self, ec_number: str, verbose: bool = False) -> pd.DataFrame:
        def download() -> list[Sequence]:
            return [
                Sequence(
                    first_accession_code=i["firstAccessionCode"],
                    naa=int(i["noOfAminoAcids"]),
                    sequence=i["sequence"],
                    source=i["source"],
                    organism=i["organism"],
                    id=i["id"],
                )
                for i in Client(self.wsdl).service.getSequence(
                    self.email,
                    self.password,
                    f"ecNumber*{ec_number}",
                    "sequence*",
                    "noOfAminoAcids*",
                    "firstAccessionCode*",
                    "source*",
                    "id*",
                    "organism*",
                )
            ]

        return pd.DataFrame(
            self._cache_result(
                filename=self.tmp_dir / f"sequences-{ec_number}.json",
                obj_type=Sequence,
                download_fn=download,
                verbose=verbose,
            )
        )


brenda = Brenda(email=cred["EMAIL"], password=cred["PASSWORD"])

In [ ]:
(kms := brenda.get_km("4.1.1.39", verbose=True)).head()

In [ ]:
(seq := brenda.get_sequences("4.1.1.39", verbose=True)).head()

In [ ]:
by_substrate = {
    key: df.drop(columns=["substrate"]) for key, df in kms.groupby("substrate")
}

df = by_substrate["CO2"]
df.head()

In [ ]:
def plot_distributions(
    all_kms: pd.Series,
    organism_kms: pd.Series,
    *,
    ec: str,
    substrate: str,
    organism_name: str,
) -> tuple[qu.Figure, qu.Axis]:
    x = np.geomspace(all_kms.min(), all_kms.max(), 1001)
    y1 = gaussian_kde(all_kms)(x) | p(normalise)
    y2 = gaussian_kde(organism_kms)(x) | p(normalise)

    with plt.rc_context(
        {
            "grid.color": "0.8",
            "xtick.color": "0.8",
            "ytick.color": "0.8",
            "xtick.labelcolor": "0.3",
            "ytick.labelcolor": "0.3",
        }
    ):
        fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
        ax.set_title(f"Km - {ec} - {substrate}")
        ax.set_xlim(all_kms.min(), all_kms.max())
        ax.set_ylim(0, max(y1.max(), y2.max()) * 1.1)
        ax.set_xscale("log")

        ax.fill_between(x, y1, alpha=0.2)
        ax.fill_between(x, y2, alpha=0.2)
        ax.plot(x, y1, label="All")
        ax.plot(x, y2, label=organism_name)
        ax.legend()
        add_boxplot(ax, all_kms, "C0")
        add_boxplot(ax, organism_kms, "C1", offset=1)
        ax.grid()
        ax.set_frame_on(False)
    return fig, ax

In [ ]:
from typing import TypedDict


class Describe(TypedDict):
    count: int
    mean: float
    std: float


def describe(s: pd.Series) -> Describe:
    return {
        "count": len(s),
        "mean": s.mean(),
        "std": s.std(),
    }


def routine(
    organism: str,
    ec: str,
    substrate: str,
    lower_percentile: int = 5,
    upper_percentile: int = 95,
) -> str:
    path: Path

    tex: list[str] = []
    all_kms = df["value"]
    organism_kms = df[df["organism"] == organism]["value"]

    all_kms_filtered = all_kms[
        all_kms.between(
            cast(float, np.percentile(all_kms, lower_percentile)),
            cast(float, np.percentile(all_kms, upper_percentile)),
        )
    ]
    organism_kms_filtered = organism_kms[
        organism_kms.between(
            cast(float, np.percentile(organism_kms, lower_percentile)),
            cast(float, np.percentile(organism_kms, upper_percentile)),
        )
    ]

    print(
        stats := pd.DataFrame(
            {
                "All": describe(all_kms),
                "All (filtered)": describe(all_kms_filtered),
                f"{organism}": describe(organism_kms),
                f"{organism} (filtered)": describe(organism_kms_filtered),
            }
        ).T
    )

    tex.append(
        stats.to_latex(
            formatters={
                "count": int,
                "mean": partial(np.format_float_scientific, precision=1, trim="0"),
                "std": partial(np.format_float_scientific, precision=1, trim="0"),
            }
        )
    )

    fig, ax = plot_distributions(
        all_kms,
        organism_kms,
        ec=ec,
        substrate=substrate,
        organism_name=organism,
    )
    path = qu.savefig(fig, f"km-{ec}-before-filtering", path=TexPath / "img")
    tex.append(
        qu.tex.figure(
            path.relative_to(*path.parts[:1]),
            caption="Distribution of Km values before any filtering was applied.",
            label=f"fig:km-{ec}-before",
            width=r"0.6\linewidth",
        )
    )
    plt.show()

    fig, ax = plot_distributions(
        all_kms_filtered,
        organism_kms_filtered,
        ec=ec,
        substrate=substrate,
        organism_name=organism,
    )
    path = qu.savefig(fig, f"km-{ec}-after-filtering", path=TexPath / "img")
    tex.append(
        qu.tex.figure(
            path.relative_to(*path.parts[:1]),
            caption=(
                f"Distribution of Km values after filtering out all values below "
                fr"{lower_percentile} \% or above {upper_percentile} \% percentile."
            ),
            label=f"fig:km-{ec}-after",
            width=r"0.6\linewidth",
        )
    )
    plt.show()

    return "\n".join(tex)


tex = routine(
    organism="Nicotiana tabacum",
    ec="4.1.1.39",
    substrate="CO2",
)

In [ ]:
def export_tex_document(
    content: str,
    author: str,
    title: str = "Model construction",
) -> str:
    return rf"""\documentclass{{article}}
\usepackage[english]{{babel}}
\usepackage[a4paper,top=2cm,bottom=2cm,left=2cm,right=2cm,marginparwidth=1.75cm]{{geometry}}
\usepackage{{amsmath, amssymb, array, booktabs, breqn, caption, longtable, mathtools, ragged2e, tabularx, titlesec, titling}}
\newcommand{{\sectionbreak}}{{\clearpage}}
\setlength{{\parindent}}{{0pt}}
\title{{{title}}}
\date{{}} % clear date
\author{{{author}}}
\begin{{document}}
\maketitle
\tableofcontents

{content}
\end{{document}}
"""


with open((TexPath) / "main.tex", "w") as fp:
    fp.write(export_tex_document(tex, ""))